# Intro 05 — Discrete Random Variables and Distributions

Practice notebook for the **"Discrete Random Variables and Distributions"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Random variables and the PMF

A **random variable** \(X\) is a function from the sample space \(\Omega\) to
the real numbers \(\mathbb{R}\): it assigns each outcome a numeric value.
\(X\) is **discrete** when it takes values in a countable set, e.g.
\(\{0,1,2,3\}\).

For a discrete \(X\), the **probability mass function** (pmf) is

$$
p_X(x) = P(X = x),
$$

which must satisfy

$$
p_X(x) \geq 0 \quad\text{for all } x,
\qquad
\sum_{x} p_X(x) = 1.
$$

The PDF works the example "toss a fair coin three times, let \(X\) be the
number of heads". Then
\(X \in \{0,1,2,3\}\) with

$$
p_X(0)=\tfrac18,\quad p_X(1)=\tfrac38,\quad p_X(2)=\tfrac38,\quad p_X(3)=\tfrac18.
$$

We build that pmf by counting outcomes, plot it, and verify the two axioms.


In [ ]:
import itertools

# Three fair tosses: count heads over all 2^3 = 8 outcomes
outcomes = list(itertools.product("HT", repeat=3))
heads = [sum(1 for t in o if t == "H") for o in outcomes]

vals = [0, 1, 2, 3]
pmf = np.array([heads.count(k) for k in vals]) / len(outcomes)

print("Outcome -> #heads:")
for o, h in zip(outcomes, heads):
    print(f"  {''.join(o)} -> {h}")
print()
for k, p in zip(vals, pmf):
    print(f"p_X({k}) = {p}  (expected {[1/8,3/8,3/8,1/8][k]:.4f})")

print()
print(f"All probabilities >= 0: {(pmf >= 0).all()}")
print(f"Sum of pmf = {pmf.sum():.6f}  (should be 1)")

fig, ax = plt.subplots(figsize=(7, 5))
ax.stem(vals, pmf, basefmt=" ")
ax.set_xlabel("k (number of heads)")
ax.set_ylabel(r"$p_X(k)$")
ax.set_title("PMF of #heads in 3 fair tosses")
ax.set_xticks(vals)
plt.show()


---
## Part 2 — Bernoulli and Binomial distributions

A **Bernoulli** random variable \(X\) takes value \(1\) (success) with
probability \(p\) and \(0\) (failure) with probability \(1-p\):

$$
P(X=1)=p,\qquad P(X=0)=1-p.
$$

The **binomial** distribution models the number of successes in \(n\)
independent Bernoulli trials. If \(X \sim \text{Binomial}(n,p)\),

$$
P(X=k) = \binom{n}{k} p^k (1-p)^{n-k},\qquad k=0,1,\dots,n.
$$

The PDF computes, for \(n=5, p=0.6, k=3\),

$$
P(X=3) = \binom{5}{3}(0.6)^3(0.4)^2 \approx 0.3456.
$$

We verify this with `scipy.stats.binom` and plot the full pmf.


In [ ]:
from scipy.stats import binom

n, p = 5, 0.6
k = 3

# Closed-form
from math import comb
p_exact = comb(n, k) * p**k * (1 - p)**(n - k)
# scipy
p_scipy = binom.pmf(k, n, p)

print(f"Closed-form:  P(X={k}) = {p_exact:.6f}")
print(f"scipy.stats:  P(X={k}) = {p_scipy:.6f}")
print(f"PDF claims:   P(X=3) ≈ 0.3456")
assert np.isclose(p_exact, 0.3456, atol=1e-4)
print("Match: binomial pmf == PDF example ✔")

# Full pmf
ks = np.arange(0, n + 1)
pmf = binom.pmf(ks, n, p)

fig, ax = plt.subplots(figsize=(7, 5))
ax.stem(ks, pmf, basefmt=" ")
ax.set_xlabel("k (successes)")
ax.set_ylabel(r"$P(X=k)$")
ax.set_title(f"Binomial(n={n}, p={p}) pmf")
ax.set_xticks(ks)
plt.show()


---
## Part 3 — Geometric and Poisson distributions

The **geometric** distribution models the number of Bernoulli trials until
the first success. With success probability \(p\) per trial,

$$
P(X=k) = (1-p)^{k-1}p,\qquad k=1,2,\dots.
$$

The PDF uses \(p=0.2\): \(P(X=1)=0.2\), \(P(X=2)=0.8\cdot0.2=0.16\),
\(P(X=3)=0.8^2\cdot0.2=0.128\), and so on.

The **Poisson** distribution models the number of events in a fixed interval
when events occur independently at a constant average rate \(\lambda\). If
\(X \sim \text{Poisson}(\lambda)\),

$$
P(X=k) = \frac{e^{-\lambda}\lambda^k}{k!},\qquad k=0,1,2,\dots
$$

With \(\lambda=5\): \(P(X=0)=e^{-5}\approx0.0067\), and
\(P(X=5)=\frac{e^{-5}5^5}{5!}\approx0.175\). We reproduce both pmfs with
`scipy.stats.geom` and `scipy.stats.poisson`.


In [ ]:
from scipy.stats import geom, poisson

# --- Geometric (scipy convention: k = 1, 2, ... = number of trials) ---
p = 0.2
for k in [1, 2, 3]:
    p_cf = (1 - p)**(k - 1) * p
    p_sp = geom.pmf(k, p)
    print(f"Geometric: P(X={k}) = {p_cf:.4f}  (scipy {p_sp:.4f})")

ks_g = np.arange(1, 16)
pmf_g = geom.pmf(ks_g, p)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].stem(ks_g, pmf_g, basefmt=" ")
axes[0].set_xlabel("k (trials until first success)")
axes[0].set_ylabel(r"$P(X=k)$")
axes[0].set_title(f"Geometric(p={p}) pmf")
axes[0].set_xticks(ks_g)

# --- Poisson ---
lam = 5.0
print()
for k in [0, 5]:
    from math import factorial, exp
    p_cf = exp(-lam) * lam**k / factorial(k)
    p_sp = poisson.pmf(k, lam)
    print(f"Poisson: P(X={k}) = {p_cf:.4f}  (scipy {p_sp:.4f})")

ks_p = np.arange(0, 16)
pmf_p = poisson.pmf(ks_p, lam)

axes[1].stem(ks_p, pmf_p, basefmt=" ")
axes[1].set_xlabel("k (events in interval)")
axes[1].set_ylabel(r"$P(X=k)$")
axes[1].set_title(f"Poisson(lambda={lam:g}) pmf")
axes[1].set_xticks(ks_p)

plt.tight_layout()
plt.show()


---
## Part 4 — Expectation and variance

For a discrete \(X\) with pmf \(p(x)\),

$$
E[X] = \sum_x x\,p(x), \qquad
\operatorname{Var}(X) = E[(X - E[X])^2] = \sum_x (x - E[X])^2 p(x).
$$

For \(X \sim \text{Binomial}(n,p)\),

$$
E[X] = np, \qquad \operatorname{Var}(X) = np(1-p).
$$

We simulate a large sample from \(\text{Binomial}(n,p)\) and compare the
sample mean and (unbiased) sample variance against the theoretical values.


In [ ]:
n, p = 5, 0.6
mu_theory = n * p
var_theory = n * p * (1 - p)

rng = np.random.default_rng(42)
N = 200_000
samples = rng.binomial(n, p, size=N)

sample_mean = samples.mean()
sample_var = samples.var(ddof=1)   # unbiased

print(f"Theory:    E[X] = {mu_theory:.4f},  Var(X) = {var_theory:.4f}")
print(f"Sample:    mean = {sample_mean:.4f},  var  = {sample_var:.4f}")
print(f"Abs error: mean = {abs(sample_mean - mu_theory):.4f},  var = {abs(sample_var - var_theory):.4f}")

# Also via scipy.stats.binom mean/var
print(f"scipy:     mean = {binom.mean(n, p):.4f},  var = {binom.var(n, p):.4f}")
assert np.isclose(sample_mean, mu_theory, atol=0.02)
assert np.isclose(sample_var, var_theory, atol=0.05)
print("Match: sample mean/var ≈ theory ✔")


---
## Part 5 — Poisson approximation to the binomial

When \(n\) is large and \(p\) is small, \(\text{Binomial}(n,p)\) is well
approximated by \(\text{Poisson}(\lambda = np)\). The PDF defines the Poisson
as the "law of rare events". We plot both pmfs for large \(n\), small \(p\),
and watch them line up.


In [ ]:
n, p = 100, 0.03
lam = n * p   # = 3

ks = np.arange(0, 20)
pmf_bin = binom.pmf(ks, n, p)
pmf_poi = poisson.pmf(ks, lam)

print(f"Binomial(n={n}, p={p}): mean={binom.mean(n,p):.4f}, var={binom.var(n,p):.4f}")
print(f"Poisson(lambda={lam}):   mean={poisson.mean(lam):.4f}, var={poisson.var(lam):.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.stem(ks - 0.2, pmf_bin, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="Binomial")
ax.stem(ks + 0.2, pmf_poi, linefmt="C3--", markerfmt="C3s", basefmt=" ", label="Poisson")
ax.set_xlabel("k")
ax.set_ylabel(r"$P(X=k)$")
ax.set_title(f"Poisson approximation: Bin({n},{p}) vs Poi({lam:g})")
ax.set_xticks(ks)
ax.legend()
plt.show()

max_err = np.abs(pmf_bin - pmf_poi).max()
print(f"Max |pmf_binom - pmf_poisson| over k=0..19: {max_err:.5f}")
print(f"Total variation distance: {0.5 * np.abs(pmf_bin - pmf_poi).sum():.5f}")
assert max_err < 0.01
print("Poisson closely tracks the binomial when n large, p small ✔")


---
**Your turn:**

- In Part 1, change the coin to be **biased** (\(P(H)=0.6\)) and recompute the
  pmf of the number of heads in 3 tosses by enumeration. Does it still sum to 1?
- In Part 2, vary \(p\) from \(0.1\) to \(0.9\) and watch the binomial pmf shift
  from right-skewed to left-skewed. At what \(p\) is it symmetric?
- In Part 3, find the median of a \(\text{Geometric}(p=0.2)\) by hand and with
  `geom.median`. Why is the median smaller than the mean \(1/p\)?
- In Part 4, increase \(N\) (the simulation size) by \(10\times\). How much do
  the sample mean/variance errors shrink? This is the law of large numbers.
- In Part 5, push \(n=1000, p=0.001\) (so \(\lambda=1\)) and re-plot. Is the
  approximation even better? Try \(n=10, p=0.3\) (so \(np=3\), but \(p\) is not
  small) — does the approximation break down?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. A biased coin has \(P(H)=0.6\). Toss it 3 times and let \(X\) be the
number of heads. Enumerate the \(2^3=8\) outcomes, build the pmf of \(X\),
verify it sums to 1, and compute \(E[X]\) and \(\operatorname{Var}(X)\) by
hand (i.e. directly from the pmf, not from the binomial formulas).

2. Let \(X \sim \text{Binomial}(10, 0.4)\). Compute
\(P(X \leq 2)\) using (a) the binomial pmf term-by-term and (b) the CDF
`binom.cdf`. Also report \(E[X]\) and \(\operatorname{Var}(X)\) and check them
against a simulation of \(100\,000\) draws.

3. Calls arrive at a call center as a Poisson process with rate
\(\lambda = 4\) per minute. Compute the probability of exactly 0, 1, 2, 3
calls in a minute, and the probability of **at least 4** calls in a minute.
What is the expected number of calls in a 5-minute interval, and its
variance?

4. Each trade you attempt succeeds with probability \(p=0.2\), independently.
Let \(X\) be the number of attempts until the first success (geometric).
Compute \(P(X=1), P(X=2), P(X=3)\), the expected number of attempts \(E[X]\),
the standard deviation, and the probability you need more than 10 attempts
\(P(X>10)\) using `geom.sf`.

5. Show numerically that \(\text{Poisson}(\lambda)\) is the limit of
\(\text{Binomial}(n, p)\) as \(n \to \infty, p \to 0\) with \(np = \lambda\)
fixed. Use \(\lambda = 2\), \(n \in \{10, 100, 1000\}\) and report the maximum
absolute pmf difference over \(k = 0, \dots, 15\). Comment on the rate of
convergence.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Outcomes and head counts with \(P(H)=0.6\):

```python
import itertools
outcomes = list(itertools.product("HT", repeat=3))
heads = [sum(1 for t in o if t == "H") for o in outcomes]
vals = [0, 1, 2, 3]
probs = {0: 0.4**3, 1: 3*0.6*0.4**2, 2: 3*0.6**2*0.4, 3: 0.6**3}
pmf = np.array([probs[k] for k in vals])
print(pmf, pmf.sum())            # sums to 1
EX = (pmf * vals).sum()
VarX = (pmf * (vals - EX)**2).sum()
print(EX, VarX)                  # 1.8, 0.72 (= 3*0.6*0.4)
```

**2.**

```python
from scipy.stats import binom
n, p = 10, 0.4
# (a) term-by-term
p_le2_terms = sum(binom.pmf(k, n, p) for k in [0,1,2])
# (b) CDF
p_le2_cdf = binom.cdf(2, n, p)
print(p_le2_terms, p_le2_cdf)    # both ~0.1673
print(n*p, n*p*(1-p))            # 4.0, 2.4
rng = np.random.default_rng(0)
s = rng.binomial(n, p, 100_000)
print(s.mean(), s.var(ddof=1))    # ~4.0, ~2.4
```

**3.** For \(\text{Poisson}(4)\):

```python
from scipy.stats import poisson
lam = 4
for k in [0,1,2,3]:
    print(k, poisson.pmf(k, lam))
print("P(X>=4) =", 1 - poisson.cdf(3, lam))   # ~0.5665
# 5-minute interval: lambda_5 = 4*5 = 20
print(poisson.mean(20), poisson.var(20))      # 20, 20
```

**4.**

```python
from scipy.stats import geom
p = 0.2
for k in [1,2,3]:
    print(k, geom.pmf(k, p))   # 0.2, 0.16, 0.128
print("E[X] =", geom.mean(p))  # 1/p = 5.0
print("sd   =", geom.std(p))   # sqrt((1-p))/p ~ 4.472
print("P(X>10) =", geom.sf(10, p))  # (1-p)^10 ~ 0.1074
```

**5.**

```python
from scipy.stats import binom, poisson
lam = 2.0
ks = np.arange(0, 16)
for n in [10, 100, 1000]:
    p = lam / n
    err = np.abs(binom.pmf(ks, n, p) - poisson.pmf(ks, lam)).max()
    print(f"n={n:4d}: max abs err = {err:.6f}")
# As n grows (p shrinks), the max error shrinks like O(p) = O(1/n);
# the convergence is linear in p, so doubling n roughly halves the error.
```


</details>
